In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import RandomizedSearchCV

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [3]:
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "num",
            StandardScaler(),
            numerical_cols
        )
    ]
)

C:\Users\shara\AppData\Local\Temp\ipykernel_31676\1410071383.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns.tolist()


In [4]:
def evaluate_model(model):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:,1]

    return {
        "Accuracy":accuracy_score(y_test,y_pred),
        "Precision":precision_score(y_test,y_pred),
        "Recall":recall_score(y_test,y_pred),
        "F1":f1_score(y_test,y_pred),
        "ROC AUC":roc_auc_score(y_test,y_prob)
    }

In [5]:
lr_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",LogisticRegression(random_state=42))
])

lr_params = {

    "classifier__C":[0.01,0.1,1,10,100],

    "classifier__solver":[
        "liblinear",
        "lbfgs"
    ],

    "classifier__class_weight":[
        None,
        "balanced"
    ]
}

lr_search = RandomizedSearchCV(

    lr_pipeline,

    lr_params,

    n_iter=8,

    cv=5,

    scoring="f1",

    random_state=42,

    n_jobs=-1
)

lr_search.fit(X_train,y_train)

print("Best Parameters")

print(lr_search.best_params_)

lr_results = evaluate_model(

    lr_search.best_estimator_
)

print(lr_results)

Best Parameters
{'classifier__solver': 'lbfgs', 'classifier__class_weight': None, 'classifier__C': 100}
{'Accuracy': 0.8571428571428571, 'Precision': 0.5925925925925926, 'Recall': 0.3404255319148936, 'F1': 0.43243243243243246, 'ROC AUC': 0.8069601171504867}


In [6]:
dt_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",DecisionTreeClassifier(random_state=42))
])

dt_params={

"classifier__criterion":[
    "gini",
    "entropy"
],

"classifier__max_depth":[
    3,5,7,10,None
],

"classifier__min_samples_split":[
    2,5,10
],

"classifier__min_samples_leaf":[
    1,2,4
]
}

dt_search=RandomizedSearchCV(

dt_pipeline,

dt_params,

n_iter=10,

cv=5,

scoring="f1",

random_state=42,

n_jobs=-1

)

dt_search.fit(X_train,y_train)

print(dt_search.best_params_)

dt_results=evaluate_model(

dt_search.best_estimator_

)

print(dt_results)

{'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': None, 'classifier__criterion': 'gini'}
{'Accuracy': 0.8129251700680272, 'Precision': 0.39473684210526316, 'Recall': 0.3191489361702128, 'F1': 0.35294117647058826, 'ROC AUC': 0.6641829614953915}


In [11]:
rf_pipeline=Pipeline([
("preprocessor",preprocessor),
("classifier",RandomForestClassifier(random_state=42))
])

rf_params={

"classifier__n_estimators":[
100,
200,
300,
500
],

"classifier__max_depth":[
5,
10,
20,
None
],

"classifier__min_samples_split":[
2,
5,
10
],

"classifier__min_samples_leaf":[
1,
2,
4
],

"classifier__class_weight":[
None,
"balanced"
]
}

rf_search=RandomizedSearchCV(

rf_pipeline,

rf_params,

n_iter=15,

cv=5,

scoring="f1",

random_state=42,

n_jobs=-1

)

rf_search.fit(X_train,y_train)

print(rf_search.best_params_)

rf_results=evaluate_model(

rf_search.best_estimator_

)

print(rf_results)

{'classifier__n_estimators': 500, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': None, 'classifier__class_weight': 'balanced'}
{'Accuracy': 0.826530612244898, 'Precision': 0.4642857142857143, 'Recall': 0.5531914893617021, 'F1': 0.5048543689320388, 'ROC AUC': 0.7843052803859074}


In [12]:
xgb_pipeline=Pipeline([
("preprocessor",preprocessor),
("classifier",
XGBClassifier(
random_state=42,
eval_metric="logloss"
))
])

xgb_params={

"classifier__n_estimators":[
100,
200,
300
],

"classifier__learning_rate":[
0.01,
0.05,
0.1,
0.2
],

"classifier__max_depth":[
3,
4,
5,
6
],

"classifier__subsample":[
0.8,
1.0
],

"classifier__colsample_bytree":[
0.8,
1.0
]
}

xgb_search=RandomizedSearchCV(

xgb_pipeline,

xgb_params,

n_iter=15,

cv=5,

scoring="f1",

random_state=42,

n_jobs=-1

)

xgb_search.fit(X_train,y_train)

print(xgb_search.best_params_)

xgb_results=evaluate_model(

xgb_search.best_estimator_

)

print(xgb_results)

{'classifier__subsample': 0.8, 'classifier__n_estimators': 300, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.2, 'classifier__colsample_bytree': 0.8}
{'Accuracy': 0.8537414965986394, 'Precision': 0.6111111111111112, 'Recall': 0.23404255319148937, 'F1': 0.3384615384615385, 'ROC AUC': 0.7791368765612887}


In [9]:
comparison=pd.DataFrame({

"Model":[

"Logistic Regression",

"Decision Tree",

"Random Forest",

"XGBoost"

],

"Accuracy":[

lr_results["Accuracy"],

dt_results["Accuracy"],

rf_results["Accuracy"],

xgb_results["Accuracy"]

],

"Precision":[

lr_results["Precision"],

dt_results["Precision"],

rf_results["Precision"],

xgb_results["Precision"]

],

"Recall":[

lr_results["Recall"],

dt_results["Recall"],

rf_results["Recall"],

xgb_results["Recall"]

],

"F1":[

lr_results["F1"],

dt_results["F1"],

rf_results["F1"],

xgb_results["F1"]

],

"ROC AUC":[

lr_results["ROC AUC"],

dt_results["ROC AUC"],

rf_results["ROC AUC"],

xgb_results["ROC AUC"]

]

})

comparison.sort_values(

"F1",

ascending=False

)

,Model,Accuracy,Precision,Recall,F1,ROC AUC
2,Random Forest,0.826531,0.464286,0.553191,0.504854,0.784305
0,Logistic Regression,0.857143,0.592593,0.340426,0.432432,0.806960
1,Decision Tree,0.812925,0.394737,0.319149,0.352941,0.664183
3,XGBoost,0.853741,0.611111,0.234043,0.338462,0.779137


Use the Best Model

In [13]:
best_model = rf_search.best_estimator_

In [14]:
y_prob = best_model.predict_proba(X_test)[:, 1]

In [16]:
thresholds = np.arange(0.20, 0.75, 0.05)

results = []

for threshold in thresholds:

    y_pred = (y_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

results = pd.DataFrame(results)

results

,Threshold,Accuracy,Precision,Recall,F1
0,0.20,0.401361,0.202765,0.936170,0.333333
1,0.25,0.527211,0.235632,0.872340,0.371041
2,0.30,0.663265,0.300000,0.829787,0.440678
3,0.35,0.741497,0.356436,0.765957,0.486486
4,0.40,0.768707,0.376471,0.680851,0.484848
5,0.45,0.806122,0.424242,0.595745,0.495575
6,0.50,0.826531,0.464286,0.553191,0.504854
7,0.55,0.829932,0.465116,0.425532,0.444444
8,0.60,0.846939,0.531250,0.361702,0.430380
9,0.65,0.836735,0.476190,0.212766,0.294118


In [17]:
results.sort_values(
    by="F1",
    ascending=False
)

,Threshold,Accuracy,Precision,Recall,F1
6,0.50,0.826531,0.464286,0.553191,0.504854
5,0.45,0.806122,0.424242,0.595745,0.495575
3,0.35,0.741497,0.356436,0.765957,0.486486
4,0.40,0.768707,0.376471,0.680851,0.484848
7,0.55,0.829932,0.465116,0.425532,0.444444
2,0.30,0.663265,0.300000,0.829787,0.440678
8,0.60,0.846939,0.531250,0.361702,0.430380
1,0.25,0.527211,0.235632,0.872340,0.371041
0,0.20,0.401361,0.202765,0.936170,0.333333
9,0.65,0.836735,0.476190,0.212766,0.294118


In [18]:
best_threshold = results.loc[
    results["F1"].idxmax(),
    "Threshold"
]

print(best_threshold)

0.49999999999999994


"Although a threshold of 0.50 achieved the highest F1-score, a threshold of 0.35 was selected for deployment because it increased Recall from 55.3% to 76.6%. In an employee attrition use case, identifying more at-risk employees is more valuable than minimizing false positives, making the lower threshold more aligned with business objectives."

In [19]:
import joblib

# Save the tuned pipeline
joblib.dump(
    rf_search.best_estimator_,
    "../models/attrition_pipeline.pkl"
)

print("Pipeline saved successfully!")

Pipeline saved successfully!


In [20]:
best_threshold = 0.35

joblib.dump(
    best_threshold,
    "../models/threshold.pkl"
)

print("Threshold saved successfully!")

Threshold saved successfully!
